# CodeSwitch-TTS: A Hybrid Non-Autoregressive Draft and Autoregressive Verification Framework for Code-Mixed Tamil-English Speech Synthesis

This notebook runs the unified **CodeSwitch-TTS** speculative decoding pipeline designed for Tanglish (Tamil-English code-mixed speech) that achieves **<300ms Time-To-First-Audio (TTFA)** on NVIDIA T4 GPUs.

### Instructions:
1. **Accelerator**: Select **GPU T4 x2** (Notebook Settings -> Accelerator -> GPU T4).
2. **Hugging Face Token** (Optional for faster downloads): In the top menu, go to **Add-ons -> Secrets**, add a secret with label  and your Hugging Face token.
3. **Run All**: Click **Run All** to install staged dependencies, load the hybrid pipeline, synthesize audio, and benchmark latency.


## 1. Hardware & System Diagnostics

In [ ]:
import subprocess, sys, torch
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
else:
    print("Running on CPU mode. Please enable GPU T4 in Kaggle notebook settings for full acceleration.")


## 2. Hugging Face Authentication (via Kaggle Secrets)

In [ ]:
from huggingface_hub import login

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    if hf_token:
        login(token=hf_token)
        print("✓ Hugging Face login successful via Kaggle Secrets!")
    else:
        print("ℹ HF_TOKEN not set in Kaggle Secrets; proceeding with anonymous downloads.")
except Exception as e:
    print(f"ℹ Proceeding with public access ({e}).")


## 3. Clone Repository & Staged Dependency Installation

In [ ]:
import os, sys, importlib.metadata as md

REPO_URL = "https://github.com/tripathiji1312/codeSwitch.git"
CLONE_DIR = "/kaggle/working/codeSwitch"

# --- 1. Always sync the repository (fast-forward the clone, or clone fresh) ---
if os.path.exists(os.path.join(CLONE_DIR, ".git")):
    if os.system(f"git -C {CLONE_DIR} fetch origin main") != 0:
        os.system(f"rm -rf {CLONE_DIR}")
        os.system(f"git clone {REPO_URL} {CLONE_DIR}")
    else:
        os.system(f"git -C {CLONE_DIR} reset --hard FETCH_HEAD")
else:
    os.system(f"rm -rf {CLONE_DIR}")
    os.system(f"git clone {REPO_URL} {CLONE_DIR}")

# --- 2. Decide from the installed state (not a marker file) whether to (re)install ---
def _deps_ok() -> bool:
    try:
        if tuple(map(int, md.version("protobuf").split(".")[:2])) < (5, 26):
            return False                        # no google.protobuf.runtime_version
        if md.version("transformers") != "4.46.1":
            return False
        md.version("parler-tts")                # raises if parler-tts is missing
        md.version("sentencepiece")
        md.version("jiwer")                     # evaluation suite (CER/WER)
        md.version("matplotlib")                # paper figures
        return True
    except Exception:
        return False

if not _deps_ok():
    print("Installing dependencies (staged to avoid package conflicts)...")
    # Stage 1: parler-tts from GitHub (brings its transformers/audiotools chain)
    os.system("pip install -q -U git+https://github.com/huggingface/parler-tts.git")
    # Stage 2: pin transformers to the exact version parler-tts v0.2.2 targets
    os.system("pip install -q 'transformers==4.46.1'")
    # Stage 3: remaining audio/scientific libs
    os.system("pip install -q descript-audio-codec soundfile scipy accelerate sentencepiece")
    # Stage 3b: evaluation-suite extras (CER/WER + IEEE-style figures)
    os.system("pip install -q jiwer matplotlib seaborn pandas")
    # Stage 4: protobuf LAST and >= 5.26 (it ships google.protobuf.runtime_version).
    # parler-tts -> descript-audiotools pins protobuf<5 and silently downgrades it to
    # 4.25.9, which broke the Parler-TTS import and forced the noise-only fallback.
    os.system("pip install -q 'protobuf==5.29.5'")
    print("✓ Dependencies installed!")
    print("⚠ The kernel now restarts so the new protobuf/transformers take effect.")
    print("👉 Click 'Run All' again — installation is skipped on the second pass.")
    os._exit(0)  # hard restart; a stale protobuf 4.x cannot be swapped in-process
else:
    print("✓ Dependencies already installed.")

print(f"   protobuf:     {md.version('protobuf')}  (needs >= 5.26)")
print(f"   transformers: {md.version('transformers')}")

# --- 3. Sanity-check the Parler-TTS import (the real cause of the static-noise runs) ---
try:
    from parler_tts import ParlerTTSForConditionalGeneration  # noqa: F401
    print("✓ parler_tts import OK")
except Exception as e:
    print(f"⚠ parler_tts import FAILED: {type(e).__name__}: {e}")
    print("  Fix: !pip install -q 'protobuf==5.29.5'  then restart the session (Stop & Run All).")

# --- 4. Make the packaged code importable ---
for path_candidate in [CLONE_DIR, ".", ".."]:
    if os.path.exists(os.path.join(path_candidate, "codeswitch_tts")):
        if path_candidate not in sys.path:
            sys.path.insert(0, path_candidate)
        print(f"✓ Root path configured: {path_candidate}")
        break


## 4. Initialize CodeSwitch-TTS Hybrid Pipeline

In [ ]:
import torch
from codeswitch_tts.config import CodeSwitchTTSConfig
from codeswitch_tts.engine.streaming_pipeline import ChunkedStreamingPipeline
from codeswitch_tts.utils.audio import save_wav
from IPython.display import Audio, display

print("Initializing CodeSwitch-TTS...")
config = CodeSwitchTTSConfig()
pipeline = ChunkedStreamingPipeline(config=config)

if getattr(pipeline, "speech_available", False):
    print("✓ Pipeline ready! Authentic Indic Parler-TTS speech backend is active.")
else:
    print("⚠ Indic Parler-TTS did not load: synthesis would only produce random-weight")
    print("  noise. Apply the fix printed above, restart the session, and re-run.")


## 5. Streaming Synthesis on 3 Research Test Utterances with Audio Playback

In [ ]:
test_utterances = [
    {
        "category": "Native Tamil",
        "text": "உங்கள் பயணம் வெற்றிகரமாக முடிந்தது.",
        "filename": "sample1_native_tamil.wav",
    },
    {
        "category": "Romanized Tanglish",
        "text": "Unga cab ippo pickup location-ku vandhuchu.",
        "filename": "sample2_romanized_tanglish.wav",
    },
    {
        "category": "Transit Code-Mixed",
        "text": "Driver-ku OTP 5291 sollunga, total fare Rs. 245.",
        "filename": "sample3_transit_codemixed.wav",
    },
]

os.makedirs("output_audio", exist_ok=True)
benchmark_results = []

# ---------------------------------------------------------------------------
# Warm-up pass.  The first synthesis after loading the model pays one-off CUDA
# kernel / allocator / description-cache costs (on a T4: ~4.3 s vs ~0.7 s once
# warm - which previously made the first benchmarked category look ~20x slower
# than the others).  Two throwaway utterances - including a long code-mixed one
# so the DAC decoder is exercised at full length - are synthesised with timings
# discarded, so the benchmark below reports steady-state numbers.
# ---------------------------------------------------------------------------
print("Warming up (CUDA kernels, allocator, DAC, description cache)...")
for warm_text in (
    "இது ஒரு வெப்பநிலை சோதனை, முடிவுகளில் சேர்க்கத் தேவையில்லை.",
    "Warm-up only: driver-ku OTP 1234 sollunga, total fare Rs. 500, vehicle TN 09 AB 1234.",
):
    try:
        pipeline.synthesize(warm_text)
    except Exception as exc:
        print(f"  [warmup] skipped: {type(exc).__name__}: {exc}")
print("Warm-up done (timings discarded).")
print("=" * 75)


for i, sample in enumerate(test_utterances, 1):
    print("=" * 75)
    print(f" [{i}/3] Category: {sample['category']}")
    print(f" Raw Input:      \"{sample['text']}\"")
    
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        
    result = pipeline.synthesize(sample["text"])
    benchmark_results.append((sample, result))
    
    print(f" Normalized:     \"{result.normalized_text}\"")
    print(f" Detected Reg:   {result.detected_language}")
    print(f" TTFA (ms):      {result.ttfa_ms:.1f} ms  (Target: <300 ms)")
    print(f" Total Latency:  {result.total_latency_ms:.1f} ms")
    print(f" Audio Duration: {result.duration_sec:.2f} s")
    print(f" RTF:            {result.real_time_factor:.3f}")
    print(f" CB0 Acceptance: {result.cb0_acceptance_rate:.1f}%")
    
    # Save Audio
    out_path = os.path.join("output_audio", sample["filename"])
    save_wav(out_path, result.audio_waveform, sample_rate=result.sample_rate)
    print(f" ✓ Saved to: {out_path}")
    
    # Display inline interactive audio player
    print(f" Listen ({sample['category']}):")
    display(Audio(result.audio_waveform, rate=result.sample_rate))


## 6. Latency Analysis & Architectural Comparison Table

In [ ]:
print("=" * 80)
print(" BENCHMARK SUMMARY & LATENCY ANALYSIS")
print("=" * 80)
header = f"{'Category':<22} | {'TTFA (ms)':<10} | {'Total (ms)':<11} | {'Audio (s)':<10} | {'RTF':<8} | {'CB0 Accept %'}"
print(header)
print("-" * 80)
for sample, res in benchmark_results:
    print(f"{sample['category']:<22} | {res.ttfa_ms:<10.1f} | {res.total_latency_ms:<11.1f} | {res.duration_sec:<10.2f} | {res.real_time_factor:<8.3f} | {res.cb0_acceptance_rate:.1f}%")
print("-" * 80)

ttfas = [res.ttfa_ms for _, res in benchmark_results]
if len(ttfas) >= 2:
    warm = ttfas[1:]
    print(f" TTFA breakdown: first measured utterance = {ttfas[0]:.1f} ms | "
          f"warm mean = {sum(warm) / len(warm):.1f} ms | warm min = {min(warm):.1f} ms")
    print(" (cold-start overhead was absorbed by the warm-up cell above)")
    print("-" * 80)

print()
print("For the paper: run the full suite (cells below) - 60 utterances x 3 configurations,")
print("true (teacher-forced) + unrelated-target reference CB0 acceptance per linguistic")
print("zone, IEEE figures and LaTeX tables (I-III) written to paper_figures/.")
print()
print("ARCHITECTURAL LATENCY COMPARISON - DESIGN TARGETS (not measured here):")
print(" ┌───────────────────────────────────────┬──────────────────┬──────────────────┐")
print(" │ Metric                                │ AR Baseline      │ CodeSwitch-TTS   │")
print(" ├───────────────────────────────────────┼──────────────────┼──────────────────┤")
print(" │ First Chunk TTFA                      │ ~3,200 ms        │ ~230 - 260 ms    │")
print(" │ Speculative Decoding Codebook Target  │ All 9 CBs (Fails)│ Codebook 0 Only  │")
print(" │ Codebook Rejection Collapse Risk      │ High (>77% rej.) │ Target: low rej. │")
print(" │ Residual Codebook 1-8 Generation      │ Sequential AR    │ Parallel NAR     │")
print(" │ Real-Time Factor (RTF)                │ ~1.20 (Lagging)  │ ~0.24 (target)   │")
print(" │ TTFA Latency Reduction Target         │ Baseline (1.0x)  │ >92% (13.6x fast)│")
print(" └───────────────────────────────────────┴──────────────────┴──────────────────┘")
print(" ⚠ The box above lists plan targets; quote the measured TTFA / CB0 acceptance from")
print("   the table above and from paper_figures/tables.tex, never the targets as results.")


## 7. Interactive Custom Tanglish Synthesis

In [ ]:
# Test any custom Tanglish or transit text here:
custom_text = "Driver anna, pickup location reach aana udane call pannunga, OTP 4821."

print(f'Input Text: "{custom_text}"')
custom_res = pipeline.synthesize(custom_text)
print(f'Normalized: "{custom_res.normalized_text}"')
print(f"TTFA:       {custom_res.ttfa_ms:.1f} ms")
print(f"Acceptance: {custom_res.cb0_acceptance_rate:.1f}%")
display(Audio(custom_res.audio_waveform, rate=custom_res.sample_rate))


## 8a. Build a Register-Balanced Distillation Corpus

The benchmark is 20 native Tamil / 20 romanised Tanglish / 20 transit code-mixed
utterances, but the template generator's *unique* output is ~81 % romanised (pure
Tamil templates carry almost no slots). This script mixes in entity/OTP/fare
insertion, rule-based romanisation and real Tamil text, dedupes, guarantees
disjointness from the eval set and then sub-samples each register to a target quota.

**Size is the budget knob**: every corpus line costs ≈ 10 s of teacher time on a T4 (measured:
2 000 lines in 5 h 36 m), so `--max-sentences` is how the corpus is made to fit the session —
`2000` ≈ 5.5 h, `3900` (uncapped) ≈ 11 h. The cap keeps the 34/33/33 mix, and selection is prefix-stable, so
raising the cap later only *adds* sentences (the teacher cache is content-addressed and
only collects the new ones).

* `--templates 2500 --wikipedia 900` → 2 000 lines, 34/33/33, ~5.5 h of teacher time
* `--wikipedia 0` → no internet needed (fewer real Tamil sentences, same mix)
* `--wikipedia-file tamil.txt` → reuse a local/cached dump offline
* `--extra-file a.txt b.json` → your own sentences (txt: one per line, json: list or
  `[{"text": ...}]`)


In [ ]:
import os
os.chdir('/kaggle/working/codeSwitch')
# 2000 lines at 34/33/33 - the file is what the teacher pass pays for, so --max-sentences
# is the knob that makes this fit one T4 session (~5.5 h of teacher time).
!python build_codeswitch_corpus.py --templates 2500 --wikipedia 900 --max-sentences 2000 \
    --out codeswitch_corpus.txt --json-out codeswitch_corpus.stats.json


## 8. Self-Distillation of the Draft / Residual Models

**Two phases, deliberately separate.** The teacher pass is the bottleneck (measured ≈ 10 s/sentence);
running it alone means a session that dies at 90 % keeps the whole collection, and the draft /
residual training is then a short, wall-clock-capped step on a finished cache.

* Teacher shards are cached **by sentence content** (`teacher_cache/shard_<sha1>.npz`), so
  Ctrl-C is always safe, a second session continues where this one stopped, and growing the
  corpus only collects the new lines. The collection line prints
  `cache: 812/2000 sentences already collected (1188 to go, ~2h36m)` up front.
* `--target-source auto` captures the LM tokens and validates them against the teacher's own
  audio (prints `len_ratio`/`corr`); it falls back to DAC re-encoding if that fails.
* The phase-1 cell ends with a **W&B artifact backup** of the cache (it is only a few MB), so
  a dead session is one `python push_artifacts_to_wandb.py --download teacher --dest .` away;
  Kaggle *Quick Save* is not a reliable file backup.
* The residual uses **scheduled sampling** (`--residual-sched 0.5`): with a probability that
  ramps 0 → 0.5 it is trained on the *draft's* CB0 instead of the teacher's, which is the
  distribution it actually sees at inference time.


In [ ]:
import os
os.chdir('/kaggle/working/codeSwitch')
# ---- phase 1: teacher only (~5.5 h for 2000 lines; Ctrl-C / session death is safe) ----
# The run ends with an automatic W&B backup of the cache (last lines of this cell) —
# that backup is the recovery path if the session later dies. Then continue with phase 2.
# --num-sentences must match the corpus size — the trainer silently caps at 300 without it.
!python train_draft_distill.py --corpus-json codeswitch_corpus.txt \
    --num-sentences 2000 --skip-draft --skip-residual --ckpt-dir codeswitch_ckpt

# Back up the finished teacher cache to W&B (this is the ~5.5 h a dead session would eat).
# Recover in any fresh session with:  !python push_artifacts_to_wandb.py --download teacher --dest .
!pip install -q wandb
!python push_artifacts_to_wandb.py --group teacher


In [ ]:
import os
os.chdir('/kaggle/working/codeSwitch')
# ---- phase 2: draft + residual on the finished cache (~30-50 min wall-clock) ----
# --min-epochs 300 keeps early stopping from ending training before the wall-clock caps
# bind (the best-val checkpoint is still restored), so a noisy val split cannot leave the
# student undertrained again.
# --num-sentences must match the corpus size — the trainer silently caps at 300 without it.
!python train_draft_distill.py --corpus-json codeswitch_corpus.txt \
    --num-sentences 2000 --draft-epochs 60 --residual-sched 0.5 --min-epochs 300 \
    --max-minutes 30 --max-minutes-residual 20 --ckpt-dir codeswitch_ckpt
# watch for a `[Residual] windows cover ...` line: it logs frames the 24-frame minimum
# drops; with the tail-window fix it should be absent or tiny on 2000-line shards.

# Back up the trained student (draft + residual + meta.json) to W&B as well:
#   !python push_artifacts_to_wandb.py --download student --dest .
!pip install -q wandb
!python push_artifacts_to_wandb.py --group student


## 8b. CB0-only diagnostic (decision point, ~10 min)

Synthesises **one** utterance with config C and, per streaming chunk, saves two wavs:

* `chunkN_cb0only.wav` — draft + rejection sampling only (codebooks 1-8 muted): is the *text* intelligible?
* `chunkN_full.wav` — what config C actually sounds like end to end.

It also prints the new `codeswitch_ckpt/meta.json` (draft val top-1 vs chance; residual
`cb_top1`). Read the pairs before spending time on the benchmark:

* cb0-only intelligible, full = noise → the residual decoder is still the weak point.
* **both** noise → the draft itself is failing at this scale: stop and send the wavs before
  spending eval time.

The patched `decode_codes` mutes codebooks 1-8 on the same decode path the engine uses, so
the comparison isolates exactly the residual contribution.


In [ ]:
# ===== CB0-only diagnostic (Kaggle, run in the SAME session as the eval run) =====
# Answers: is the draft+rejection-sampled Codebook-0 audio intelligible, and are the
# last 8 codebooks (residual decoder) the thing that turns B/C into "mic rubbing" noise?
# Runs config C on ONE utterance with ASR/UTMOS off. ~5-8 min (model load dominates).
import os, sys, json, pathlib

for cand in ("/kaggle/working/codeSwitch", "codeSwitch", "."):
    if os.path.isfile(os.path.join(cand, "run_full_evaluation.py")):
        os.chdir(cand)
        break
else:
    raise SystemExit("could not find run_full_evaluation.py — cd to the repo first")
sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())

import run_full_evaluation as R
from codeswitch_tts.utils.audio import save_wav

OUT = pathlib.Path("/kaggle/working/cb0diag")
OUT.mkdir(parents=True, exist_ok=True)
CKPT = "codeswitch_ckpt"

# --- Step 0: what the 2000-sentence checkpoints actually learned --------
meta_path = pathlib.Path(CKPT) / "meta.json"
if meta_path.is_file():
    print(json.dumps(json.loads(meta_path.read_text()), indent=2)[:4000])
else:
    print("no meta.json; ckpt dir:", sorted(p.name for p in pathlib.Path(CKPT).iterdir()))

# --- Step 1: capture each decoded chunk + a CB1-8-muted twin -------------
_orig_decode = R.EvalSynthesizer.decode_codes
MAX_CHUNKS = 10
_saved = 0

def _patched_decode_codes(self, codes):
    global _saved
    wave = _orig_decode(self, codes)   # unchanged: this is the audio you heard
    try:
        if _saved < MAX_CHUNKS and codes.dim() == 2 and codes.shape[0] == 9:
            idx = _saved + 1
            cb0_only = codes.clone()
            cb0_only[1:] = 0                     # mute codebooks 1..8 (constant token 0)
            wave_cb0 = _orig_decode(self, cb0_only)   # SAME decode path as above
            save_wav(OUT / f"chunk{idx}_full.wav", wave, 44100)
            save_wav(OUT / f"chunk{idx}_cb0only.wav", wave_cb0, 44100)
            _saved = idx
            print(
                f"[diag] chunk {idx}: {codes.shape[-1]} frames | "
                f"token range {int(codes.min())}..{int(codes.max())} | "
                f"saved -> cb0diag/chunk{idx}_full.wav + chunk{idx}_cb0only.wav"
            )
    except Exception as exc:
        print(f"[diag] capture skipped on this chunk ({type(exc).__name__}: {exc})")
    return wave

R.EvalSynthesizer.decode_codes = _patched_decode_codes

try:
    rc = R.main([
        "--configs", "C",
        "--limit", "1",                 # first utterance only
        "--gate-thresholds", "0",       # the un-gated hybrid (the noisy config)
        "--skip-asr", "--no-utmos",     # keep it fast; measurement not needed here
        "--ckpt", CKPT,
        "--results", str(OUT / "raw_results.jsonl"),
        "--audio-dir", str(OUT / "audio"),
        "--figures-dir", str(OUT / "figs"),
    ])
    print("main() ->", rc)
except Exception as exc:
    print("main() raised:", type(exc).__name__, exc)

print(f"captured {_saved} chunk pairs")
print("wavs:", sorted(p.name for p in OUT.glob("*.wav")))


In [ ]:
from IPython.display import Audio, display
import glob

pairs = sorted(glob.glob('/kaggle/working/cb0diag/chunk*_full.wav'))
print(f"{len(pairs)} chunk pair(s) in /kaggle/working/cb0diag")
for p in pairs[:4]:
    twin = p.replace('_full.wav', '_cb0only.wav')
    print('FULL     - what config C sounds like:', p.split('/')[-1])
    display(Audio(p))
    print('CB0-ONLY - residual muted:', twin.split('/')[-1])
    display(Audio(twin))


## 9. Full Benchmark Evaluation (60 utterances × 3 configs)

Runs Config **A** (pure AR, real `ParlerTTSStreamer`), **B** (pure NAR) and
**C** (CodeSwitch-TTS hybrid speculative) on the imported 60-utterance Tanglish
benchmark, then writes IEEE-style figures and LaTeX tables (I–V) into the fresh `paper_figures_2k/`:

* `fig1` latency/RTF/CER distributions, `fig2` per-zone acceptance, `fig3` streaming
  timeline, `fig4` length/register scaling, `fig5` gate Pareto
* Table I medians carry 95 % bootstrap CIs; CER is also reported script-robust
  (`CER_lat`, both sides transliterated to Latin); UTMOS22 and MCD (vs. Pure AR) are
  computed from the saved wavs

**ASR judge**: `--asr-model openai/whisper-large-v3` — the big-run upgrade from whisper-small,
which looped on unintelligible audio and inflated CER (ASR generation is capped by audio
length now too). The first run downloads the model (~3 GB).

**Gate sweep** — default `--gate-mode derived --gate-thresholds auto`: C is synthesised once
at τ=0 and the whole confidence-gate curve is derived from the measured C and Pure-AR rows.
The gate decides *before* synthesis, so the mixture is exact for quality/acceptance and a
fallback costs exactly the two measured passes — the curve is therefore free (no extra GPU
time). `auto` uses quantiles of the confidence the draft actually shows, which guarantees
interior operating points; fixed thresholds such as `0.6,0.75,0.9` fall back on *every*
utterance when the draft's mean top-1 is ≈0.1. Use `--gate-mode synth --gate-thresholds
0,0.3,0.34,0.36` to synthesise the mixture end to end — the run then also prints a
derived-vs-synthesised validation line.

Add `--wandb` to push the finished run to Weights & Biases (cell 9b does it separately).

Results are appended atomically after every utterance — just re-run the cell to resume.
Existing `(config, utterance, gate)` rows are **skipped**, so after a code update you can
either re-run this cell (fast, nothing new) or refresh tables/figures from the saved
JSONL at zero GPU cost with:
```python
!python run_full_evaluation.py --replay
```
Budget **~30-60 min** on a T4 from scratch (whisper-large-v3 ASR included; the derived gate
sweep adds no synthesis).

**Provenance guard**: `eval_results_2k/provenance.json` records a hash of the checkpoints the
existing rows came from. If you retrain and point the evaluation at the same `--results` file,
the run stops (exit 3) instead of mixing two models' audio — pass `--force` to re-synthesise,
or use a different `--results` path to keep both runs.


In [ ]:
import os
os.chdir('/kaggle/working/codeSwitch')
!python run_full_evaluation.py --ckpt codeswitch_ckpt \
    --asr-model openai/whisper-large-v3 \
    --results eval_results_2k/raw_results.jsonl \
    --audio-dir eval_audio_2k --figures-dir paper_figures_2k


## 9b. Push the Run to Weights & Biases

Store your key once — Kaggle **Add-ons → Secrets → Add** a secret named `WANDB_API_KEY` (turn
the notebook's *Internet* switch on) — the cell picks it up automatically (`--api-key`,
`$WANDB_API_KEY`, Kaggle secret, in that order).

One W&B run gets everything the paper needs:

* **scalars** — TTFA-c warm/cold, TTFA-e2e, total latency, RTF, CER / `CER_lat`, UTMOS, MCD,
  CB0 acceptance (true + mismatched reference), target-distribution log-likelihoods, the whole
  gate sweep (`gate/*/tau_*`), distance-to-switch buckets and the training metrics from
  `codeswitch_ckpt/meta.json` — grouped into panels for run-to-run comparison
* **figures** — fig1–fig5 as images (PDFs ride in the artifact)
* **tables** — configs / gate sweep / distance-to-switch / per-register as W&B tables
* **audio** — best/median/worst-CER sample per config, captioned with utterance, register and CER
* **artifact** — `raw_results.jsonl`, `tables.tex`, `summary.json`, `run_meta.json`, the figures,
  the audio and the checkpoint meta, so a wiped Kaggle session never loses a run

Re-run this cell after `--replay` to refresh the upload at zero GPU cost. `--run-id <id>
--group c2500` overwrites/regroups an existing run, and `--dry-run` lists what would be
uploaded without touching the network.


In [ ]:
import os
os.chdir('/kaggle/working/codeSwitch')
!pip install -q wandb
# WANDB_API_KEY comes from the Kaggle secret of the same name.
!python push_to_wandb.py --project codeswitch-tts --tags kaggle,t4 \
    --figures-dir paper_figures_2k --audio-dir eval_audio_2k --ckpt-dir codeswitch_ckpt \
    --results eval_results_2k/raw_results.jsonl --max-audio 4


## 10. Collect Artifacts

Bundles the figures, tables, metrics and audio samples for download.

The same bundle (plus the raw JSONL and the checkpoint meta) lives in the W&B artifact from
cell 9b, so a disconnected or wiped session never loses the numbers - `wandb` keeps them
versioned and downloadable.


In [ ]:
import os
os.chdir('/kaggle/working/codeSwitch')
# Figures + tables + metrics + audio samples + CB0-only diagnostic wavs (weights excluded)
!zip -qr /kaggle/working/codeswitch_artifacts_2k.zip paper_figures_2k eval_results_2k eval_audio_2k codeswitch_ckpt cb0diag -x 'codeswitch_ckpt/*.pt'
!ls -lh /kaggle/working/codeswitch_artifacts_2k.zip

# Back the final zip up to W&B too (last safety net):
#   !python push_artifacts_to_wandb.py --download bundle --dest /kaggle/working
!pip install -q wandb
!python push_artifacts_to_wandb.py --group bundle --zip /kaggle/working/codeswitch_artifacts_2k.zip
